In [2]:
from collections import Counter, defaultdict
import json
from pathlib import Path
import shutil

# Student Metadata
STUDENT_ID = "TUPM-26-1836"
LAST_NAME = "Blanca"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

# Parameters: First 1026 records, Target status class 4xx
sample_size = 1000 + ID_SUM
target_status_class = str((SEED_DIGIT % 4) + 2)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "weblog.csv"

total_records_processed = 0
target_class_records = []
status_counts = Counter()

with open(dataset_path, "r", encoding="utf-8", errors="ignore") as file:
  header = file.readline().strip().split(",")
  header_map = {name.strip(): idx for idx, name in enumerate(header)}

  ip_idx = header_map.get("IP", 0)
  time_idx = header_map.get("Time", 1)
  url_idx = header_map.get("URL", 2)
  status_idx = header_map.get("Status", 3)

  for _ in range(sample_size):
    line = file.readline()
    if not line:
      break
    total_records_processed += 1

    fields = line.strip().split(",")
    if len(fields) > max(ip_idx, time_idx, url_idx, status_idx):
      ip, timestamp, url, status = (
          fields[ip_idx],
          fields[time_idx],
          fields[url_idx],
          fields[status_idx],
      )
      status_counts[status] += 1

      if status.startswith(target_status_class):
        target_class_records.append(
            {"IP": ip, "Time": timestamp, "URL": url, "Status": status}
        )

all_target_urls = Counter(rec["URL"] for rec in target_class_records)
top_5_urls = all_target_urls.most_common(5)

filtered_csv_path = (
    OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered_weblog.csv"
)
with open(filtered_csv_path, "w", encoding="utf-8") as csv_out:
  csv_out.write("IP,Time,URL,Status\n")
  for rec in target_class_records:
    csv_out.write(f"{rec['IP']},{rec['Time']},{rec['URL']},{rec['Status']}\n")

json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"
summary_data = {
    "student_id": STUDENT_ID,
    "last_name": LAST_NAME,
    "records_processed": total_records_processed,
    "target_status_class": f"{target_status_class}xx",
    "records_in_target_class": len(target_class_records),
    "status_counts": dict(status_counts),
    "top_5_frequent_urls": top_5_urls,
}

with open(json_summary_path, "w", encoding="utf-8") as json_out:
  json.dump(summary_data, json_out, indent=4)

backup_json_path = BACKUP_DIR / json_summary_path.name
shutil.copy2(json_summary_path, backup_json_path)

print(f"Processed: {total_records_processed}, Target Class: {target_status_class}xx")
print(f"Filtered CSV: {filtered_csv_path.exists()}, Backup Size: {backup_json_path.stat().st_size} bytes")

Processed: 1026, Target Class: 4xx
Filtered CSV: True, Backup Size: 495 bytes


In [3]:
import csv
import json
from pathlib import Path
import shutil

STUDENT_ID = "TUPM-26-1836"
LAST_NAME = "Blanca"
SEED_DIGIT = int(STUDENT_ID[-1])

# Parameters: OS is Android (since SEED_DIGIT 6 is even), Behavior Class is (6 % 5) + 1 = 2
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
target_behavior_class = (SEED_DIGIT % 5) + 1

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"

total_records = 0
matching_records = []
total_battery_drain = 0.0
total_data_usage = 0.0

with open(dataset_path, "r", encoding="utf-8", errors="ignore") as file:
  reader = csv.DictReader(file)
  for row in reader:
    total_records += 1
    # Adjust key names based on actual dataset column headers if they vary slightly
    os_val = row.get("Operating System", "").strip()
    behavior_val = row.get("User Behavior Class", "").strip()

    if os_val == target_os and behavior_val == str(target_behavior_class):
      try:
        battery = float(row.get("Battery Drain (mAh/day)", 0))
        data_use = float(row.get("Data Usage (MB/day)", 0))
        total_battery_drain += battery
        total_data_usage += data_use

        matching_records.append({
            "User ID": row.get("User ID"),
            "Device Model": row.get("Device Model"),
            "Operating System": os_val,
            "Battery Drain (mAh/day)": battery,
            "Data Usage (MB/day)": data_use,
        })
      except ValueError:
        continue

count = len(matching_records)
avg_battery = total_battery_drain / count if count > 0 else 0.0
avg_data = total_data_usage / count if count > 0 else 0.0

filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_filtered.csv"
with open(filtered_csv_path, "w", encoding="utf-8", newline="") as csv_out:
  fieldnames = [
      "User ID",
      "Device Model",
      "Operating System",
      "Battery Drain (mAh/day)",
      "Data Usage (MB/day)",
  ]
  writer = csv.DictWriter(csv_out, fieldnames=fieldnames)
  writer.writeheader()
  writer.writerows(matching_records)

json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
summary_data = {
    "target_os": target_os,
    "target_behavior_class": target_behavior_class,
    "total_records_in_dataset": total_records,
    "matching_records_count": count,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data, 2),
}

with open(json_summary_path, "w", encoding="utf-8") as json_out:
  json.dump(summary_data, json_out, indent=4)

backup_csv_path = BACKUP_DIR / filtered_csv_path.name
shutil.copy2(filtered_csv_path, backup_csv_path)

print(f"Target OS: {target_os}, Behavior Class: {target_behavior_class}")
print(f"Matching Records: {count}, Avg Battery: {avg_battery:.2f}")
print(f"Backup File Size: {backup_csv_path.stat().st_size} bytes")

Target OS: Android, Behavior Class: 2
Matching Records: 114, Avg Battery: 887.11
Backup File Size: 4573 bytes


In [4]:
import csv
import json
from pathlib import Path
import shutil

STUDENT_ID = "TUPM-26-1836"
LAST_NAME = "Blanca"
SEED_DIGIT = int(STUDENT_ID[-1])

# Parameters: Target Year is 2015 + 6 = 2021
target_year = 2015 + SEED_DIGIT

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"

total_records = 0
target_incidents = []
attack_type_counts = {}
total_loss = 0.0

with open(dataset_path, "r", encoding="utf-8", errors="ignore") as file:
  reader = csv.DictReader(file)
  for row in reader:
    total_records += 1
    # Adjust column headers based on exact CSV column names if needed
    year_str = row.get("Year", "").strip()
    attack_type = row.get("Attack Type", "Unknown").strip()
    loss_str = row.get("Financial Loss (in Million $)", "0").strip()

    try:
      record_year = int(year_str) if year_str else 0
      loss_val = float(loss_str) if loss_str else 0.0
    except ValueError:
      continue

    if record_year == target_year:
      attack_type_counts[attack_type] = (
          attack_type_counts.get(attack_type, 0) + 1
      )
      total_loss += loss_val
      target_incidents.append({
          "Year": record_year,
          "Attack Type": attack_type,
          "Financial Loss": loss_val,
          "Details": row,
      })

incident_count = len(target_incidents)
avg_loss = total_loss / incident_count if incident_count > 0 else 0.0

# Sort by financial loss descending, tie-breaking by original order
target_incidents.sort(key=lambda x: x["Financial Loss"], reverse=True)
top_5_costliest = target_incidents[:5]

json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_top5.json"
with open(json_summary_path, "w", encoding="utf-8") as json_out:
  json.dump(top_5_costliest, json_out, indent=4)

filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
with open(filtered_csv_path, "w", encoding="utf-8", newline="") as csv_out:
  if target_incidents:
    fieldnames = target_incidents[0]["Details"].keys()
    writer = csv.DictWriter(csv_out, fieldnames=fieldnames)
    writer.writeheader()
    for inc in target_incidents:
      writer.writerow(inc["Details"])

backup_json_path = BACKUP_DIR / json_summary_path.name
shutil.copy2(json_summary_path, backup_json_path)

report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
with open(report_path, "w", encoding="utf-8") as rep:
  rep.write("CYBERSECURITY INCIDENT REPORT\n")
  rep.write("-" * 30 + "\n")
  rep.write(f"Target Year: {target_year}\n")
  rep.write(f"Incidents in Target Year: {incident_count}\n")
  rep.write(f"Total Financial Loss: ${total_loss:,.2f}M\n")
  rep.write(f"Average Financial Loss: ${avg_loss:,.2f}M\n")
  rep.write(f"Attack Type Counts: {attack_type_counts}\n")

print(f"Target Year: {target_year}, Incidents Found: {incident_count}")
print(f"Total Loss: ${total_loss:,.2f}M, Report Created: {report_path.name}")

Target Year: 2021, Incidents Found: 299
Total Loss: $15,873.41M, Report Created: Blanca_TUPM-26-1836_cyber_report.txt
